In [ ]:
# Importing packages
import os
from pathlib import Path

import pandas as pd
import numpy as np

import scanpy as sc
import squidpy as sq
import anndata as ad

import json

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [ ]:
# Set working directory
project = Path("/media/nannu1375/Backpack/Shankara/KNC")

os.chdir(project)

## Sample Loading

In [ ]:
# Read the h5 files
adata = sc.read_10x_h5(
    filename=project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/filtered_feature_bc_matrix.h5"
    )

In [ ]:
# Read the tissue positions file
positions = pd.read_parquet(
    project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/spatial/tissue_positions.parquet"
)

In [ ]:
# Join the positions file with the anndata file
## We are going to merge the positions file with obs first, then take the x y coordinates and put it in obsm since it just needs a matrix of coordinates.
adata.obs = adata.obs.join(
    positions.set_index("barcode")
)

# create the obsm x-y coordinates
adata.obsm["spatial"] = adata.obs[
    ["pxl_col_in_fullres", "pxl_row_in_fullres"]
].to_numpy()

In [ ]:
# read the scale factors
with open(
    project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/spatial/scalefactors_json.json"
) as f:
    scalefactors = json.load(f)

# Read the h&e images
hires = mpimg.imread(
    project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/spatial/tissue_hires_image.png"
)

lowres = mpimg.imread(
    project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/spatial/tissue_lowres_image.png"
)

# Display the image
plt.imshow(hires)
scalefactors

In [ ]:
# Build the uns dictionary
adata.uns["spatial"] = {
    "Visium_HD": {
        "images":{
            "hires": hires,
            "lowres": lowres,
        },
        "scalefactors": scalefactors,
        "metadata": {}
    }
}

## Sanity checks

In [ ]:
# =============================================================================
# Sanity checks for the Spatial AnnData object
# =============================================================================

print("=" * 70)
print("AnnData Summary")
print("=" * 70)
print(adata)

print("\n")

# =============================================================================
# Check dimensions
# =============================================================================

print("=" * 70)
print("Dimensions")
print("=" * 70)

print(f"Number of spatial bins (obs): {adata.n_obs:,}")
print(f"Number of genes (var):        {adata.n_vars:,}")

print("\n")

# =============================================================================
# Check available AnnData slots
# =============================================================================

print("=" * 70)
print("Available Slots")
print("=" * 70)

print("obsm keys :", list(adata.obsm.keys()))
print("uns keys  :", list(adata.uns.keys()))

print("\n")

# =============================================================================
# Check observation (spot/bin) metadata
# =============================================================================

print("=" * 70)
print("Observation metadata (adata.obs)")
print("=" * 70)

display(adata.obs.head())

print(f"\nShape : {adata.obs.shape}")

print("\n")

# =============================================================================
# Check gene metadata
# =============================================================================

print("=" * 70)
print("Gene metadata (adata.var)")
print("=" * 70)

display(adata.var.head())

print(f"\nShape : {adata.var.shape}")

print("\n")

# =============================================================================
# Check expression matrix
# =============================================================================

print("=" * 70)
print("Expression Matrix (adata.X)")
print("=" * 70)

print(type(adata.X))
print("Shape :", adata.X.shape)

print("\n")

# =============================================================================
# Check spatial coordinates
# =============================================================================

print("=" * 70)
print("Spatial Coordinates")
print("=" * 70)

print(type(adata.obsm["spatial"]))
print("Shape :", adata.obsm["spatial"].shape)

print("\nFirst five coordinates:")
print(adata.obsm["spatial"][:5])

print("\n")

# =============================================================================
# Check spatial metadata
# =============================================================================

print("=" * 70)
print("Spatial Metadata")
print("=" * 70)

library_id = list(adata.uns["spatial"].keys())[0]

print("Library ID :", library_id)

print("\nAvailable entries:")
print(list(adata.uns["spatial"][library_id].keys()))

print("\nScale factors:")
print(adata.uns["spatial"][library_id]["scalefactors"])

print("\n")

# =============================================================================
# Verify consistency
# =============================================================================

print("=" * 70)
print("Consistency Checks")
print("=" * 70)

assert adata.n_obs == adata.obs.shape[0], "Mismatch: obs rows"

assert adata.n_obs == adata.obsm["spatial"].shape[0], \
    "Mismatch: spatial coordinates"

assert adata.n_vars == adata.var.shape[0], \
    "Mismatch: gene metadata"

print("✓ Number of observations matches obs")
print("✓ Number of observations matches spatial coordinates")
print("✓ Number of genes matches var")

print("\nAll sanity checks passed!")

In [ ]:
# Create a copy of the raw counts
adata.layers["counts"] = adata.X.copy()

print(adata.layers["counts"])

## QC step

In [ ]:
# Annotate mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Annotate ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

adata.var["hb"] = adata.var_names.isin(hb_genes)

# Display the mitochondrial genes and how many
print("Number of mitochondrial and nuclear genes (False is nuclear and true is mitochondrial): ", adata.var["mt"].value_counts())
print("List of gene names of the mitochondrial genes:\n", adata.var_names[adata.var["mt"] == True])

# Display the ribosomal genes and how many
print("\nNumber of ribosomal and nuclear genes (False is nuclear and true is ribosomal): ", adata.var["ribo"].value_counts())
print("List of gene names of the ribosomal genes:\n", adata.var_names[adata.var["ribo"] == True])

# Display the hemoglobin genes and how many
print("\nNumber of hemoglobin and nuclear genes (False is nuclear and true is hemoglobin): ", adata.var["hb"].value_counts())
print("List of gene names of the hemoglobin genes:\n", adata.var_names[adata.var["hb"] == True])

In [ ]:
# Calculating QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    inplace=True
)